# NB10 — PyO3: Calling Rust from Python

**Module 17: HPC, Parallel Computing, and Rust**

---

## 1. Motivation

You have written Rust code that is 10–100x faster than Python. How do you use it from a Python notebook, a Django app, or a bioinformatics pipeline that expects a Python API?

**PyO3** is the bridge: it provides Rust macros that expose Rust functions as Python-callable functions, and handles type conversion between Python and Rust automatically. **Maturin** is the build tool that compiles the Rust code and installs it as a Python package with a single command: `maturin develop`.

needletail (the FASTA parser) and sourmash (metagenomic sketching) are both real-world examples of PyO3 projects used in production bioinformatics.

---

## 2. Intuition

PyO3 works by compiling your Rust code as a shared library (`.so` on Linux/Mac, `.pyd` on Windows) that Python can import. The `#[pyfunction]` macro generates the glue code that converts Python arguments to Rust types and Rust return values back to Python types.

From Python's perspective, `import compbio_rs` is just importing a module — the fact that it's compiled Rust is invisible.

The fallback pattern shown in this notebook is important: if the Rust extension is not installed (e.g., on a machine without Rust/maturin), the Python implementation is used instead. This makes your code portable without sacrificing performance on the machines where Rust is available.

---

## 3. Biological background

**needletail** (PyO3 example from One Codex):
```python
import needletail
for record in needletail.parse_fastx_file('reads.fastq.gz'):
    gc = record.gc_content()
    seq = record.sequence  # bytes
```
This Python API is backed by a Rust parser that handles gzipped FASTQ at ~1 GB/s.

The PyO3 build pipeline:
1. `cargo build --release` compiles Rust to native code
2. PyO3 generates Python-compatible C API bindings
3. `maturin develop` links the compiled code as a Python extension
4. `import needletail` (or `compbio_rs`) works as a normal Python import

---

## 4. Mathematical explanation — type conversions

PyO3 handles these conversions automatically:

| Python type | Rust type (PyO3) |
|-------------|------------------|
| `str` | `&str` or `String` |
| `int` | `i32`, `i64`, `usize`, etc. |
| `float` | `f32`, `f64` |
| `bytes` | `&[u8]` or `Vec<u8>` |
| `list` | `Vec<T>` |
| `dict` | `HashMap<K,V>` |
| `None` | `Option<T>` |
| Exception | `PyResult<T>` (= `Result<T, PyErr>`) |

The conversion is not free: there is a marshalling cost at the boundary. For very frequent small calls, this can dominate. The design principle: **batch at the boundary** — pass large arrays or strings rather than calling Rust for each character.

---

## 5. Computational explanation — the three files you need

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("NB10: PyO3 — Calling Rust from Python")
print("This notebook shows the complete PyO3 project structure.")
print("The Rust code is shown as strings; Python fallbacks are fully executable.")

In [ ]:
# --- File 1: src/lib.rs ---

lib_rs = '''
// src/lib.rs — PyO3 module exposing Rust functions to Python

use pyo3::prelude::*;
use pyo3::exceptions::PyValueError;
use std::collections::HashMap;

// ── gc_content ─────────────────────────────────────────────────────────────
#[pyfunction]
fn gc_content(sequence: &str) -> PyResult<f64> {
    if sequence.is_empty() {
        return Err(PyValueError::new_err("Empty sequence"));
    }
    let gc = sequence.bytes()
        .filter(|&b| b == b\'C\' || b == b\'G\' || b == b\'c\' || b == b\'g\')
        .count();
    Ok(gc as f64 / sequence.len() as f64)
}

// ── reverse_complement ─────────────────────────────────────────────────────
#[pyfunction]
fn reverse_complement(sequence: &str) -> PyResult<String> {
    let rc: Vec<u8> = sequence.bytes()
        .rev()
        .map(|b| match b {
            b\'A\' => b\'T\', b\'T\' => b\'A\',
            b\'C\' => b\'G\', b\'G\' => b\'C\',
            b\'a\' => b\'t\', b\'t\' => b\'a\',
            b\'c\' => b\'g\', b\'g\' => b\'c\',
            b\'N\' | b\'n\' => b,
            other => return Err(PyValueError::new_err(
                format!("Invalid base: {}", other as char)
            )),
        })
        .collect::<Result<Vec<u8>, PyErr>>()?;  // propagate the first Err
    Ok(String::from_utf8(rc).unwrap())
}

// ── hamming_distance ───────────────────────────────────────────────────────
#[pyfunction]
fn hamming_distance(a: &str, b: &str) -> PyResult<usize> {
    if a.len() != b.len() {
        return Err(PyValueError::new_err(
            format!("Sequences have different lengths: {} vs {}", a.len(), b.len())
        ));
    }
    Ok(a.bytes().zip(b.bytes()).filter(|(x, y)| x != y).count())
}

// ── The Python module ──────────────────────────────────────────────────────
#[pymodule]
fn compbio_rs(_py: Python, m: &PyModule) -> PyResult<()> {
    m.add_function(wrap_pyfunction!(gc_content, m)?)?;
    m.add_function(wrap_pyfunction!(reverse_complement, m)?)?;
    m.add_function(wrap_pyfunction!(hamming_distance, m)?)?;
    Ok(())
}
'''

print("File: src/lib.rs")
print(lib_rs)
print("Key constructs:")
print("  #[pyfunction]   — marks a Rust fn as callable from Python")
print("  PyResult<T>     — return type; Err(...) becomes a Python exception")
print("  PyValueError    — becomes Python ValueError")
print("  #[pymodule]     — marks the module registration function")
print("  wrap_pyfunction! — macro that generates the Python-callable wrapper")

In [ ]:
# --- File 2: Cargo.toml ---

cargo_toml = '''
[package]
name = "compbio_rs"
version = "0.1.0"
edition = "2021"

[lib]
# crate-type = ["cdylib"] means: compile as a shared library
# cdylib = C-compatible dynamic library = what Python can import
crate-type = ["cdylib"]

[dependencies]
pyo3 = { version = "0.21", features = ["extension-module"] }
# extension-module feature: tells PyO3 to target the Python C API

[profile.release]
opt-level = 3      # maximum optimization
lto = true         # link-time optimization — smaller binary, faster
'''

print("File: Cargo.toml")
print(cargo_toml)

In [ ]:
# --- File 3: pyproject.toml ---

pyproject_toml = '''
[build-system]
requires = ["maturin>=1.5,<2.0"]
build-backend = "maturin"

[project]
name = "compbio-rs"
version = "0.1.0"
description = "Rust-backed bioinformatics utilities for Python"
requires-python = ">=3.9"

[tool.maturin]
# maturin knows to look for src/lib.rs given the Cargo.toml
features = ["pyo3/extension-module"]
python-source = "python"   # put Python stubs/fallbacks in python/ directory
'''

build_instructions = '''
# Build and install in development mode:
pip install maturin
maturin develop

# Then in Python:
import compbio_rs
print(compbio_rs.gc_content("ATCGCG"))
# → 0.5

# Build a distributable wheel:
maturin build --release
# → builds a .whl file compatible with the current Python version
'''

print("File: pyproject.toml")
print(pyproject_toml)
print("Build instructions:")
print(build_instructions)

In [ ]:
# --- Python fallback: used when compbio_rs is not installed ---

# This is the recommended pattern for PyO3 libraries:
# Try to import the Rust extension, fall back to pure Python.

try:
    import compbio_rs
    _RUST_AVAILABLE = True
    print("Using Rust implementation (compbio_rs)")
except ImportError:
    _RUST_AVAILABLE = False
    print("compbio_rs not installed — using Python fallback")
    print("To install: pip install maturin && maturin develop")


# Pure Python fallbacks — same API as the Rust functions
import re

def gc_content(sequence: str) -> float:
    if not sequence:
        raise ValueError("Empty sequence")
    return sum(1 for b in sequence.upper() if b in ('G', 'C')) / len(sequence)

_COMP_TABLE = str.maketrans('ATCGatcgNn', 'TAGCtagcNn')

def reverse_complement(sequence: str) -> str:
    return sequence.translate(_COMP_TABLE)[::-1]

def hamming_distance(a: str, b: str) -> int:
    if len(a) != len(b):
        raise ValueError(f"Sequences have different lengths: {len(a)} vs {len(b)}")
    return sum(x != y for x, y in zip(a, b))


# Dispatch: use Rust if available, Python otherwise
if _RUST_AVAILABLE:
    _gc = compbio_rs.gc_content
    _rc = compbio_rs.reverse_complement
    _hd = compbio_rs.hamming_distance
else:
    _gc = gc_content
    _rc = reverse_complement
    _hd = hamming_distance

# Test both paths produce the same results
test_seqs = ['ATCGCGAT', 'GGGGCCCC', 'ATATATAT']
for seq in test_seqs:
    gc_py = gc_content(seq)
    gc_dispatch = _gc(seq)
    assert abs(gc_py - gc_dispatch) < 1e-10, f"GC mismatch for {seq}"
    
    rc_py = reverse_complement(seq)
    rc_dispatch = _rc(seq)
    assert rc_py == rc_dispatch, f"RC mismatch for {seq}"

print("All correctness checks: PASSED")
print(f"gc_content('ATCGCGAT') = {_gc('ATCGCGAT'):.4f}")
print(f"reverse_complement('ATCG') = {_rc('ATCG')}")
print(f"hamming_distance('ATCG', 'ATCC') = {_hd('ATCG', 'ATCC')}")

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: PyO3 build pipeline
ax = axes[0]
ax.axis('off')
pipeline_text = (
    "PyO3 Build Pipeline\n\n"
    "  src/lib.rs\n"
    "  (Rust source with\n"
    "   #[pyfunction] annotations)\n"
    "       |\n"
    "       v  rustc + LLVM\n"
    "  native object code\n"
    "       |\n"
    "       v  PyO3 glue code\n"
    "  C API bindings generated\n"
    "       |\n"
    "       v  linker\n"
    "  compbio_rs.so  (shared lib)\n"
    "       |\n"
    "       v  maturin develop\n"
    "  installed in site-packages\n"
    "       |\n"
    "       v  Python\n"
    "  import compbio_rs\n"
    "  compbio_rs.gc_content('ATCG')\n\n"
    "One command: maturin develop"
)
ax.text(0.05, 0.98, pipeline_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#eaf2f8', alpha=0.8))
ax.set_title('PyO3 Build Pipeline', fontweight='bold')

# Panel 2: Python-Rust data flow
ax = axes[1]
ax.axis('off')
dataflow_text = (
    "Python ↔ Rust Data Flow\n\n"
    "Python call:\n"
    "  compbio_rs.gc_content('ATCG')\n\n"
    "  1. PyO3 converts str → &str\n"
    "     (zero-copy for ASCII strings)\n\n"
    "  2. Rust function executes:\n"
    "     gc_content(&str) → f64\n"
    "     (C speed, no GIL held)\n\n"
    "  3. PyO3 converts f64 → float\n"
    "     (trivial boxing)\n\n"
    "  4. Python receives: 0.5\n\n"
    "Marshalling cost:\n"
    "  Small (str→&str is near-zero)\n"
    "  Amortized over computation\n\n"
    "GIL handling:\n"
    "  PyO3 releases GIL by default\n"
    "  inside Rust code → threads work!"
)
ax.text(0.05, 0.98, dataflow_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#fef9e7', alpha=0.8))
ax.set_title('Python ↔ Rust Interop\nData Flow', fontweight='bold')

# Panel 3: Speedup expectations table
ax = axes[2]
ax.axis('off')
speedup_text = (
    "Expected Speedup: Python vs Rust\n"
    "(from needletail / noodles benchmarks)\n\n"
    "Operation         Speedup\n"
    "─────────────────────────────\n"
    "GC content        3–10x\n"
    "Reverse complement 3–5x\n"
    "Hamming distance  5–20x\n"
    "K-mer counting    5–20x\n"
    "FASTA parsing     10–30x\n"
    "FASTQ (gzipped)   10–20x\n\n"
    "Notes:\n"
    "- Lower bound: simple ops where\n"
    "  Python str.translate is fast\n"
    "- Upper bound: complex loops,\n"
    "  many allocations in Python\n"
    "- Marshalling cost negligible\n"
    "  for inputs >= 100 bases\n"
    "- Scales linearly with input\n"
    "  (Rust avoids GC pauses)"
)
ax.text(0.05, 0.98, speedup_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#eafaf1', alpha=0.8))
ax.set_title('Speedup Expectations\n(Python vs PyO3/Rust)', fontweight='bold')

plt.tight_layout()
plt.savefig('../datasets/nb10_pyo3.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Exercises

1. **Extend the fallback:** Add a `kmer_count(sequence: str, k: int) -> dict` function to the Python fallback module. The signature should match what a PyO3 Rust function would expose. Test that it works correctly for k=3, k=4, k=21.

2. **Error handling:** The Rust `reverse_complement` raises `PyValueError` for invalid bases. Write the Python fallback to raise the same error with the same message format. Test by passing 'ATCGXYZ'.

3. **Type conversion quiz:** For each Python type below, write the corresponding PyO3 Rust type and explain any conversion cost:
   - `list[str]`
   - `bytes`
   - `dict[str, int]`
   - `Optional[float]`

4. **Read needletail source:** Find `src/sequence.rs` in the needletail repository (https://github.com/onecodex/needletail/blob/master/src/sequence.rs). Identify one `#[pyfunction]` or equivalent. What are the input and output types? What Python exception would be raised on error?

## 12. Reflection

*Write here: When does the PyO3 marshalling cost make Rust acceleration not worth it? (Hint: think about the ratio of marshalling time to computation time.) For what sequence length does calling `gc_content` via PyO3 break even with a pure Python implementation, approximately?*

---

## Papers referenced

- Kryuchkova-Mostacci et al. (2025). "A Survey of Bioinformatics Tools in Rust." — now read this paper.

## References

- PyO3 User Guide: https://pyo3.rs/
- Maturin: https://www.maturin.rs/
- needletail (production PyO3 example): https://github.com/onecodex/needletail
- sourmash (Rust + PyO3 metagenomics): https://github.com/sourmash-bio/sourmash

## Future work / open questions

- PyO3 can expose Python classes (not just functions) via `#[pyclass]`. How would you expose a `FastaRecord` struct as a Python class with Python properties?
- maturin can cross-compile wheels for multiple platforms. How does this work, and why does it matter for distributing bioinformatics tools?